# RoadGuard AI — Pothole Detection
## Task 2: End-to-End Pipeline — Tracking, Severity Scoring & Automated Reporting

This notebook takes the **YOLOv8s model fine-tuned in Notebook 1** and builds it into a
real, usable pipeline:

1. **Batch image prediction** with confidence filtering and severity tagging.
2. **Multi-object tracking** on dash-cam video using ByteTrack — so the *same* pothole
   seen across 30 consecutive frames is counted **once**, not 30 times.
3. **Severity scoring** — a simple, explainable heuristic that turns bounding-box geometry
   into a Low / Medium / High repair-priority label.
4. **FPS measurement** — can this run live on patrol-vehicle hardware?
5. **Automated CSV report** — something a road-maintenance team could actually act on.

> **GPU is optional here.** Inference with YOLOv8s is fast even on CPU.
> Run this in the **same Colab session** as Notebook 1, or in a fresh session
> (just mount Drive to access your trained model).

## 0 · Environment Setup

In [ ]:
import subprocess, sys

# Install dependencies
print("Installing packages...")
subprocess.run([sys.executable, '-m', 'pip', 'install', 'ultralytics', '-q'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'opencv-python-headless', '-q'], check=True)
print("Done.")

In [ ]:
import os, glob, time, json, shutil, random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image

import torch
from ultralytics import YOLO

print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
print("Setup OK ✓")

## 1 · Load Trained Model & Dataset

This cell tries multiple locations for your `best.pt` — whether you're in the same
Colab session as Notebook 1, a fresh session with Drive mounted, or running locally.

In [ ]:
# ── Mount Google Drive ──────────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print("Google Drive mounted.")
except Exception:
    IN_COLAB = False
    print("Not in Colab — running locally.")

# ── Find the trained model ──────────────────────────────────
MODEL_SEARCH_PATHS = [
    # 1. Google Drive (saved during training in Notebook 1)
    '/content/drive/MyDrive/RoadGuardAI/runs/pothole_yolov8s/weights/best.pt',
    # 2. Same Colab session as Notebook 1 (training just finished)
    'runs/detect/pothole_yolov8s/weights/best.pt',
    # 3. Local copy downloaded from Notebook 1
    'pothole_best.pt',
    # 4. Inside app folder (Streamlit deployment structure)
    'app/weights/pothole_best.pt',
    'weights/pothole_best.pt',
]

MODEL_PATH = None
for p in MODEL_SEARCH_PATHS:
    if os.path.exists(p):
        MODEL_PATH = p
        break

# Also search for nested path bug from earlier runs
if MODEL_PATH is None:
    for root, dirs, files in os.walk('/content/drive/MyDrive/RoadGuardAI/runs'):
        if 'best.pt' in files:
            MODEL_PATH = os.path.join(root, 'best.pt')
            break

if MODEL_PATH is None:
    print("ERROR: Could not find pothole_best.pt anywhere.")
    print("Options:")
    print("  1. If in Colab: mount Drive and check /content/drive/MyDrive/RoadGuardAI/runs/")
    print("  2. Upload pothole_best.pt manually (sidebar → upload icon)")
    print("  3. Run Notebook 1 first to train the model.")
else:
    print(f"Model found: {MODEL_PATH}")
    print(f"Size: {os.path.getsize(MODEL_PATH)/1e6:.1f} MB")
    model = YOLO(MODEL_PATH)
    print(f"Classes: {model.names}")
    print("Model loaded ✓")

In [ ]:
# ── Find the dataset ────────────────────────────────────────
# Needed for the image-prediction demo (uses validation images as test inputs)
import zipfile, urllib.request

DATASET_DIR = None
for candidate in ['/content/pothole_dataset_v8', 'pothole_dataset_v8']:
    if os.path.isdir(candidate):
        DATASET_DIR = candidate
        break

if DATASET_DIR is None:
    print("Dataset not found locally. Downloading (this takes a few minutes)...")
    DATASET_ZIP = 'pothole_dataset_v8.zip'
    if not os.path.exists(DATASET_ZIP):
        urllib.request.urlretrieve(
            'https://www.dropbox.com/s/qvglw8pqo16769f/pothole_dataset_v8.zip?dl=1',
            DATASET_ZIP
        )
    with zipfile.ZipFile(DATASET_ZIP) as z:
        z.extractall('./')
    DATASET_DIR = 'pothole_dataset_v8'

val_images = sorted(glob.glob(os.path.join(DATASET_DIR, 'valid', 'images', '*')))
print(f"Dataset: {DATASET_DIR}")
print(f"Validation images available: {len(val_images)}")
print("Dataset ready ✓")

## 2 · Severity Scoring Heuristic

We don't have ground-truth pothole depth or area in metric units (that would need stereo
cameras or LiDAR), so we use a **practical, explainable proxy**: how much of the camera
frame the detected bounding box occupies.

A pothole filling a large fraction of the frame is either physically large or very close
to the vehicle — both are reasons to flag it as higher priority for repair.

| Severity | Relative bbox area | Meaning |
|---|---|---|
| 🟢 Low | < 1.5% of frame | Small or distant |
| 🟠 Medium | 1.5% – 5% of frame | Noticeable damage |
| 🔴 High | > 5% of frame | Large / close — prioritize repair |

This is deliberately simple and transparent — the kind of thing you explain and justify
in your report's "Challenges" section: a production system would calibrate this against
camera height, lens distortion, and possibly fuse in GPS-tagged road class data.

In [ ]:
def severity_from_bbox(x1, y1, x2, y2, frame_w, frame_h):
    """Classify pothole severity based on relative bounding-box area."""
    box_area   = (x2 - x1) * (y2 - y1)
    frame_area = float(frame_w * frame_h)
    rel_area   = box_area / frame_area

    if rel_area >= 0.05:
        return 'High', rel_area
    elif rel_area >= 0.015:
        return 'Medium', rel_area
    else:
        return 'Low', rel_area

# BGR colours for cv2 drawing
SEVERITY_COLORS_BGR = {
    'Low':    (0, 200, 0),    # green
    'Medium': (0, 165, 255),  # orange
    'High':   (0, 0, 255),    # red
}

# Quick sanity check
print("Small box  →", severity_from_bbox(0, 0, 30, 30, 640, 480)[0])    # Low
print("Medium box →", severity_from_bbox(0, 0, 100, 80, 640, 480)[0])   # Medium
print("Large box  →", severity_from_bbox(0, 0, 250, 200, 640, 480)[0])  # High
print("\nSeverity function OK ✓")

## 3 · Batch Image Prediction Pipeline

Run the fine-tuned model on a folder of images, filter by confidence,
tag each detection with severity, draw colour-coded boxes, and save annotated outputs.
This is what a patrol-vehicle batch-processing system would do nightly.

In [ ]:
def run_image_pipeline(image_paths, model, conf=0.40, save_dir='outputs/images'):
    """
    Runs detection on a list of images.
    Returns: annotated images saved to save_dir, DataFrame of all detections.
    """
    os.makedirs(save_dir, exist_ok=True)
    records = []
    annotated_paths = []

    for img_path in image_paths:
        img = cv2.imread(img_path)
        if img is None:
            print(f"  Warning: could not read {img_path}, skipping.")
            continue
        h, w = img.shape[:2]

        result = model(img_path, conf=conf, verbose=False)[0]

        for box in result.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            confidence = float(box.conf[0])
            severity, rel_area = severity_from_bbox(x1, y1, x2, y2, w, h)
            color = SEVERITY_COLORS_BGR[severity]

            # Draw box
            cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
            label = f"{severity} ({confidence:.2f})"
            cv2.putText(img, label, (int(x1), max(int(y1) - 8, 14)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

            records.append({
                'image':      os.path.basename(img_path),
                'confidence': round(confidence, 3),
                'severity':   severity,
                'rel_area':   round(rel_area, 4),
                'bbox_x1': round(x1, 1), 'bbox_y1': round(y1, 1),
                'bbox_x2': round(x2, 1), 'bbox_y2': round(y2, 1),
            })

        out_path = os.path.join(save_dir, os.path.basename(img_path))
        cv2.imwrite(out_path, img)
        annotated_paths.append(out_path)

    return pd.DataFrame(records), annotated_paths

print("Pipeline function defined ✓")

In [ ]:
# Run on validation images as a demo
# (In a real deployment, this would be a folder of dashcam captures from today's patrol)
random.seed(42)
sample_images = random.sample(val_images, min(12, len(val_images)))

print(f"Running detection on {len(sample_images)} images (conf=0.40) ...")
detections_df, annotated_paths = run_image_pipeline(sample_images, model, conf=0.40)

print(f"\nTotal detections: {len(detections_df)}")
if len(detections_df):
    print(f"Severity breakdown:")
    for sev in ['High', 'Medium', 'Low']:
        count = (detections_df['severity'] == sev).sum()
        if count > 0:
            print(f"  {sev:8s}: {count}")

detections_df

In [ ]:
# Visualise annotated results
n_show = min(8, len(annotated_paths))
cols   = 4
rows   = int(np.ceil(n_show / cols))

fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
axes = axes.flatten()

for i, path in enumerate(annotated_paths[:n_show]):
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    axes[i].imshow(img)
    axes[i].set_title(os.path.basename(path), fontsize=8)
    axes[i].axis('off')

for i in range(n_show, len(axes)):
    axes[i].axis('off')

plt.suptitle(
    "Batch Image Detection — severity-coded: 🟢 Low  🟠 Medium  🔴 High",
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.show()

## 4 · Video Tracking Pipeline — Counting Unique Potholes

This is the core of Task 2. A dash-cam driving past a pothole will trigger detections
across many consecutive frames — naively counting raw detections wildly over-counts the
same pothole.

We use Ultralytics' built-in **ByteTrack** tracker (`model.track(..., persist=True)`).
It assigns a stable **track ID** to each pothole across frames, so we report *unique*
potholes encountered along a route, not raw detection counts.

### Getting a test video

**Option A** — Record 10-20 seconds on your phone of any road with visible damage.  
**Option B** — Download a short dashcam clip from YouTube.  
**Option C** — Skip this section and demo tracking in the Streamlit app locally instead.

Upload your video to Colab:
```python
from google.colab import files
uploaded = files.upload()
VIDEO_PATH = list(uploaded.keys())[0]
```

In [ ]:
# ── Set your video path here ────────────────────────────────
VIDEO_PATH = 'sample_road_video.mp4'   # ← change this to your uploaded filename

# Check if video exists; if not, offer upload
if not os.path.exists(VIDEO_PATH):
    print(f"'{VIDEO_PATH}' not found.")
    print()
    print("To upload a video in Colab:")
    print("  Option 1: Use the file browser (folder icon in left sidebar) → upload")
    print("  Option 2: Run this in a new cell:")
    print("    from google.colab import files")
    print("    uploaded = files.upload()")
    print("    VIDEO_PATH = list(uploaded.keys())[0]")
    print()
    print("Then re-run this cell after setting VIDEO_PATH to the uploaded filename.")
    VIDEO_EXISTS = False
else:
    cap = cv2.VideoCapture(VIDEO_PATH)
    fps_in     = cap.get(cv2.CAP_PROP_FPS) or 25
    total_frm  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    vid_w      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    vid_h      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration   = total_frm / fps_in
    cap.release()
    print(f"Video found: {VIDEO_PATH}")
    print(f"  Resolution : {vid_w}x{vid_h}")
    print(f"  FPS        : {fps_in:.1f}")
    print(f"  Duration   : {duration:.1f}s ({total_frm} frames)")
    VIDEO_EXISTS = True

In [ ]:
def run_tracking_pipeline(video_path, model, conf=0.40, max_seconds=30,
                          out_path='outputs/tracked_output.mp4'):
    """
    Runs multi-object tracking on a video.
    Returns: per-track DataFrame, summary dict, path to annotated video.
    """
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    fps_in     = cap.get(cv2.CAP_PROP_FPS) or 25
    vid_w      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    vid_h      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    max_frames = int(fps_in * max_seconds)
    cap.release()

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(out_path, fourcc, fps_in, (vid_w, vid_h))

    track_records = {}   # track_id → best observation (highest confidence)
    frame_times   = []
    frame_idx     = 0

    print(f"Processing up to {max_frames} frames ({max_seconds}s) ...")

    results_stream = model.track(
        source  = video_path,
        conf    = conf,
        tracker = 'bytetrack.yaml',
        persist = True,
        stream  = True,
        verbose = False,
    )

    for result in results_stream:
        if frame_idx >= max_frames:
            break

        t0    = time.time()
        frame = result.orig_img.copy()
        boxes = result.boxes

        if boxes is not None and boxes.id is not None and len(boxes):
            ids   = boxes.id.int().tolist()
            xyxys = boxes.xyxy.cpu().numpy()
            confs = boxes.conf.cpu().numpy()

            for i, track_id in enumerate(ids):
                x1, y1, x2, y2 = xyxys[i]
                confidence = float(confs[i])
                severity, rel_area = severity_from_bbox(x1, y1, x2, y2, vid_w, vid_h)
                color = SEVERITY_COLORS_BGR[severity]

                # Draw on frame
                cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
                label = f"#{track_id} {severity}"
                cv2.putText(frame, label, (int(x1), max(int(y1) - 8, 14)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

                rec = {
                    'track_id':         track_id,
                    'first_seen_frame': frame_idx,
                    'confidence':       round(confidence, 3),
                    'severity':         severity,
                    'rel_area':         round(rel_area, 4),
                }
                # Keep the highest-confidence observation per track
                if track_id not in track_records or confidence > track_records[track_id]['confidence']:
                    track_records[track_id] = rec

        writer.write(frame)
        frame_times.append(time.time() - t0)
        frame_idx += 1

        # Progress every 100 frames
        if frame_idx % 100 == 0:
            print(f"  Frame {frame_idx}/{max_frames} ...")

    writer.release()
    print(f"  Processed {frame_idx} frames.")

    avg_fps = 1.0 / (np.mean(frame_times) + 1e-9) if frame_times else 0.0

    report_df = (
        pd.DataFrame(track_records.values())
        .sort_values('first_seen_frame')
        .reset_index(drop=True)
    ) if track_records else pd.DataFrame()

    summary = {
        'video':                os.path.basename(video_path),
        'total_frames':         frame_idx,
        'unique_potholes':      len(report_df),
        'avg_inference_fps':    round(avg_fps, 1),
        'severity_breakdown':   report_df['severity'].value_counts().to_dict() if len(report_df) else {},
    }

    return report_df, summary, out_path

print("Tracking pipeline function defined ✓")

In [ ]:
# ── Run tracking (only if video was found) ──────────────────
if VIDEO_EXISTS:
    MAX_SECONDS = 30   # process up to 30s — adjust if your clip is longer

    report_df, summary, tracked_video_path = run_tracking_pipeline(
        VIDEO_PATH, model, conf=0.40, max_seconds=MAX_SECONDS
    )

    print()
    print("=" * 50)
    print("  TRACKING RESULTS")
    print("=" * 50)
    print(f"  Video              : {summary['video']}")
    print(f"  Frames processed   : {summary['total_frames']}")
    print(f"  Unique potholes    : {summary['unique_potholes']}")
    print(f"  Avg inference FPS  : {summary['avg_inference_fps']}")
    print(f"  Severity breakdown : {summary['severity_breakdown']}")
    print("=" * 50)

    if len(report_df):
        display(report_df)
else:
    print("Skipped — no video provided. Set VIDEO_PATH above and re-run.")
    print("You can still demo tracking in the Streamlit app locally.")

## 5 · Automated Report — Exportable for a Maintenance Team

A CSV a road-maintenance department could actually act on:
each row = one unique pothole, with its track ID, confidence, severity, and the frame it first appeared in.

In [ ]:
os.makedirs('outputs', exist_ok=True)

# ── Image detection report ──────────────────────────────────
if len(detections_df):
    img_report_path = 'outputs/image_detection_report.csv'
    detections_df.to_csv(img_report_path, index=False)
    print(f"Image detection report saved: {img_report_path}  ({len(detections_df)} rows)")
else:
    print("No image detections to save.")

# ── Video tracking report ───────────────────────────────────
if VIDEO_EXISTS and len(report_df):
    vid_report_path = 'outputs/road_damage_report.csv'
    report_df.to_csv(vid_report_path, index=False)
    print(f"Video tracking report saved : {vid_report_path}  ({len(report_df)} rows)")

    # Severity breakdown chart
    sev_order = ['High', 'Medium', 'Low']
    sev_colors = ['#d62728', '#ff7f0e', '#2ca02c']
    counts = report_df['severity'].value_counts().reindex(sev_order).fillna(0)

    fig, ax = plt.subplots(figsize=(5, 4))
    counts.plot(kind='bar', color=sev_colors, ax=ax, edgecolor='white', linewidth=0.5)
    ax.set_title('Detected Pothole Severity Breakdown (Video)')
    ax.set_ylabel('Unique potholes')
    ax.set_xlabel('')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
elif VIDEO_EXISTS:
    print("No potholes tracked in the video at the current confidence threshold.")
else:
    print("No video was provided — tracking report skipped.")

print()
print("Reports ready for download ✓")

In [ ]:
# Download reports (Colab only)
try:
    from google.colab import files
    if os.path.exists('outputs/image_detection_report.csv'):
        files.download('outputs/image_detection_report.csv')
    if os.path.exists('outputs/road_damage_report.csv'):
        files.download('outputs/road_damage_report.csv')
    if VIDEO_EXISTS and os.path.exists(tracked_video_path):
        files.download(tracked_video_path)
    print("Downloads started — check your browser's Downloads folder.")
except ImportError:
    print("Not in Colab — find reports in the outputs/ folder.")

## 6 · Performance Analysis — Real-Time Feasibility

For deployment on a patrol vehicle, the pipeline must process frames fast enough to
not miss potholes at driving speed. This section measures actual inference throughput.

In [ ]:
# Benchmark on a batch of images (gives a stable FPS reading, works even without a video)
bench_images = random.sample(val_images, min(20, len(val_images)))

times = []
for img_path in bench_images:
    t0 = time.time()
    _ = model(img_path, conf=0.40, verbose=False)
    times.append(time.time() - t0)

avg_time = np.mean(times)
avg_fps  = 1.0 / avg_time

device = "GPU (" + torch.cuda.get_device_name(0) + ")" if torch.cuda.is_available() else "CPU"

print("=" * 50)
print("  INFERENCE SPEED BENCHMARK")
print("=" * 50)
print(f"  Device           : {device}")
print(f"  Images tested    : {len(bench_images)}")
print(f"  Avg per image    : {avg_time*1000:.1f} ms")
print(f"  Throughput       : {avg_fps:.1f} FPS")
print("=" * 50)
print()

if avg_fps >= 25:
    print("Real-time capable (≥25 FPS) — suitable for live dashcam processing.")
elif avg_fps >= 10:
    print("Near-real-time (10-25 FPS) — suitable for offline batch processing or slow driving.")
else:
    print(f"Below real-time ({avg_fps:.1f} FPS) — suitable for batch processing of saved footage.")
print(f"  (At typical driving speed of 40 km/h, 10 FPS covers ~1 frame per meter of road.)")

## 7 · (Optional) Export for Edge Deployment

If you want to go further — deploy on a Raspberry Pi, phone, or embedded device —
Ultralytics can export to ONNX / TFLite / OpenVINO directly.

In [ ]:
EXPORT_FORMATS = []  # e.g. ['onnx'] or ['onnx', 'tflite']

if EXPORT_FORMATS:
    for fmt in EXPORT_FORMATS:
        exported = model.export(format=fmt)
        print(f"Exported {fmt} → {exported}")
else:
    print("No export requested.")
    print("To try: set EXPORT_FORMATS = ['onnx'] and re-run this cell.")

## 8 · Prepare Model for Streamlit App (Local Deployment)

Download `pothole_best.pt` and place it in `app/weights/` on your laptop.
Then follow the README to run:

```bash
cd road_damage_ai/app
pip install -r requirements.txt
streamlit run streamlit_app.py
```

That app wraps everything in this notebook behind a simple web UI —
image upload, video tracking, severity scoring, report download —
usable by someone with zero Python knowledge, running entirely on CPU.

In [ ]:
# Ensure a local copy exists for download
if not os.path.exists('pothole_best.pt'):
    shutil.copy(MODEL_PATH, 'pothole_best.pt')
    print(f"Copied model to ./pothole_best.pt")

print(f"Model size: {os.path.getsize('pothole_best.pt')/1e6:.1f} MB")
print()
print("Download this file and place it at:")
print("  road_damage_ai/app/weights/pothole_best.pt")
print()

try:
    from google.colab import files
    files.download('pothole_best.pt')
    print("Download started.")
except ImportError:
    print(f"Not in Colab — model is at: {os.path.abspath('pothole_best.pt')}")

## 9 · Task 2 Summary

Fill in with your actual numbers from the outputs above:

| Item | Value |
|---|---|
| Tracking algorithm | ByteTrack (via `model.track()`) |
| Severity logic | Relative bbox area thresholds (Low < 1.5% < Medium < 5% < High) |
| Image detection report | `outputs/image_detection_report.csv` |
| Video tracking report | `outputs/road_damage_report.csv` |
| Unique potholes (demo video) | *(Section 4 output)* |
| Measured FPS | *(Section 6 output)* |
| Deployment | Streamlit web app — runs on CPU, no GPU required |

### What this pipeline adds over bare detection (Task 1)

1. **Tracking** — same pothole counted once, not N times across frames.
2. **Severity scoring** — explainable repair-priority labels (Low/Medium/High).
3. **Automated reporting** — structured CSV output a maintenance team could ingest.
4. **Performance measurement** — can it run live? (FPS benchmark answers this).
5. **Deployable interface** — Streamlit app wraps everything for non-technical users.

### What this demonstrates for the assignment

- End-to-end pipeline from trained model → inference → post-processing → report.
- Real-time performance measurement (FPS).
- Integration with a simple web dashboard (Streamlit app).
- The model was trained in Notebook 1; this notebook shows it can be *deployed and used*.